In [5]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import os

FIRE_CSV  = "/Users/hariaksha/Documents/GitHub/climate-conflict/data/climate-pressure/fire/DL_FIRE_J1VIIRS-C2_718929_Nov2000-Feb2026_buffer0km-csv/fire_archive_J1V-C2_718929.csv"
ACLED_CSV = "/Users/hariaksha/Documents/GitHub/climate-conflict/data/unrest/Indonesia_aggregated_data.csv"
SHP_L1    = "/Users/hariaksha/Documents/GitHub/climate-conflict/data/administrative/gadm41_IDN_shp/gadm41_IDN_1.shp"
SHP_L2    = "/Users/hariaksha/Documents/GitHub/climate-conflict/data/administrative/gadm41_IDN_shp/gadm41_IDN_2.shp"

os.makedirs("output", exist_ok=True)

# ---------- 1. Load shapefiles ----------
print("Loading shapefiles...")
idn1 = gpd.read_file(SHP_L1)
idn2 = gpd.read_file(SHP_L2)

# ---------- 2. Load fire data ----------
print("Loading fire data...")
fire = pd.read_csv(FIRE_CSV, usecols=["latitude", "longitude"])
fire = fire.rename(columns={"latitude": "lat", "longitude": "lon"}).dropna()
fire = fire[(fire["lat"].between(-11, 6)) & (fire["lon"].between(95, 141))]
print(f"  Fire records: {len(fire):,}")

# ---------- 3. Load and aggregate ACLED data ----------
print("Loading ACLED data...")
acled = pd.read_csv(ACLED_CSV)
acled.columns = [c.strip().upper() for c in acled.columns]

acled_agg = acled.groupby("ADMIN1")["EVENTS"].sum().reset_index()
acled_agg.columns = ["ADMIN1_en", "total_events"]

name_map = {
    "Aceh":                   "Aceh",
    "Bali":                   "Bali",
    "Banten":                 "Banten",
    "Bengkulu":               "Bengkulu",
    "Central Java":           "Jawa Tengah",
    "Central Kalimantan":     "Kalimantan Tengah",
    "Central Sulawesi":       "Sulawesi Tengah",
    "East Java":              "Jawa Timur",
    "East Kalimantan":        "Kalimantan Timur",
    "East Nusa Tenggara":     "Nusa Tenggara Timur",
    "Gorontalo":              "Gorontalo",
    "Jakarta":                "Jakarta Raya",
    "Jambi":                  "Jambi",
    "Lampung":                "Lampung",
    "Maluku":                 "Maluku",
    "North Kalimantan":       "Kalimantan Utara",
    "North Maluku":           "Maluku Utara",
    "North Sulawesi":         "Sulawesi Utara",
    "North Sumatra":          "Sumatera Utara",
    "Papua":                  "Papua",
    "Riau":                   "Riau",
    "Riau Islands":           "Kepulauan Riau",
    "South Kalimantan":       "Kalimantan Selatan",
    "South Sulawesi":         "Sulawesi Selatan",
    "South Sumatra":          "Sumatera Selatan",
    "Southeast Sulawesi":     "Sulawesi Tenggara",
    "West Java":              "Jawa Barat",
    "West Kalimantan":        "Kalimantan Barat",
    "West Nusa Tenggara":     "Nusa Tenggara Barat",
    "West Papua":             "Papua Barat",
    "West Sulawesi":          "Sulawesi Barat",
    "West Sumatra":           "Sumatera Barat",
    "Yogyakarta":             "Di Yogyakarta",
    "South Papua":            "Papua",
    "Central Papua":          "Papua",
    "Highland Papua":         "Papua",
    "Southwest Papua":        "Papua Barat",
    "Bangka Belitung":        "Bangka-Belitung",
}

acled_agg["NAME_1"] = acled_agg["ADMIN1_en"].map(name_map)
unmatched = acled_agg[acled_agg["NAME_1"].isna()]
if len(unmatched) > 0:
    print(f"  WARNING unmatched: {unmatched['ADMIN1_en'].tolist()}")

acled_agg = acled_agg.dropna(subset=["NAME_1"])
acled_final = acled_agg.groupby("NAME_1")["total_events"].sum().reset_index()
idn1 = idn1.merge(acled_final, on="NAME_1", how="left")
print(f"  Provinces matched: {idn1['total_events'].notna().sum()} / {len(idn1)}")

# ---------- 4. PLOT ----------
print("Plotting...")
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.patch.set_facecolor("white")
bounds = [95, 141, -11, 6]

island_labels = [
    ("Sumatra",    102,  0.5),
    ("Kalimantan", 114,  1.0),
    ("Java",       111, -7.2),
    ("Sulawesi",   121, -2.0),
    ("Papua",      137, -4.5),
]

for ax in axes:
    ax.set_xlim(*bounds[:2])
    ax.set_ylim(*bounds[2:])
    ax.set_facecolor("white")
    ax.tick_params(colors="black", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1.2)
    for name, x, y in island_labels:
        ax.text(x, y, name, color="#333333", fontsize=9, alpha=0.75,
                ha="center", fontweight="bold", style="italic")

# --- Map 1: Fire hexbin (district background) ---
ax1 = axes[0]
idn2.plot(ax=ax1, color="#eeeeee", edgecolor="#cccccc", linewidth=0.3)
fire_cmap = mcolors.LinearSegmentedColormap.from_list(
    "fire", ["#f5f5f5", "#fee8c8", "#fc8d59", "#d7301f", "#7f0000"])
hb = ax1.hexbin(fire["lon"], fire["lat"], gridsize=120, cmap=fire_cmap,
                mincnt=1, bins="log", extent=bounds, alpha=0.9)
cbar1 = fig.colorbar(hb, ax=ax1, fraction=0.025, pad=0.02)
cbar1.set_label("Fire Count (log scale)", color="black", fontsize=10)
cbar1.ax.yaxis.set_tick_params(color="black")
plt.setp(cbar1.ax.yaxis.get_ticklabels(), color="black")
ax1.set_title("Wildfire Detections in Indonesia\n(VIIRS Satellite Data)",
              color="black", fontsize=15, fontweight="bold", pad=12)
ax1.set_xlabel("Longitude", color="black", fontsize=10)
ax1.set_ylabel("Latitude",  color="black", fontsize=10)

# --- Map 2: Conflict choropleth (province level) ---
ax2 = axes[1]
conflict_cmap = plt.cm.YlOrRd
idn1.plot(ax=ax2, column="total_events", cmap=conflict_cmap,
          edgecolor="#aaaaaa", linewidth=0.3,
          missing_kwds={"color": "#eeeeee", "label": "No data"},
          legend=False)

sm = plt.cm.ScalarMappable(
    cmap=conflict_cmap,
    norm=mcolors.Normalize(
        vmin=idn1["total_events"].min(),
        vmax=idn1["total_events"].max()
    )
)
sm.set_array([])
cbar2 = fig.colorbar(sm, ax=ax2, fraction=0.025, pad=0.02)
cbar2.set_label("Total Conflict Events (2015–2026)", color="black", fontsize=10)
cbar2.ax.yaxis.set_tick_params(color="black")
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color="black")

no_data_patch = mpatches.Patch(color="#eeeeee", label="No conflict recorded",
                                edgecolor="#aaaaaa")
ax2.legend(handles=[no_data_patch], loc="lower left", fontsize=8,
           facecolor="white", edgecolor="#cccccc", labelcolor="black")

ax2.set_title("Conflict Intensity by Province\n(ACLED Data — Total Events 2015–2026)",
              color="black", fontsize=15, fontweight="bold", pad=12)
ax2.set_xlabel("Longitude", color="black", fontsize=10)
ax2.set_ylabel("Latitude",  color="black", fontsize=10)

# ---------- 5. Borders + spacing ----------
plt.tight_layout(pad=2.0)
plt.subplots_adjust(wspace=0.25)  # more space between panels

# Draw a black border box around each panel (including title & colorbar)
for ax in axes:
    bbox = ax.get_position()  # gets axes position in figure coords
    # Expand the box to include title above and colorbar to the right
    x0 = bbox.x0 - 0.01
    y0 = bbox.y0 - 0.01
    width  = bbox.width + 0.09   # extend right to include colorbar
    height = bbox.height + 0.10  # extend up to include title

    rect = plt.Rectangle(
        (x0, y0), width, height,
        fill=False,
        edgecolor="black",
        linewidth=2.0,
        transform=fig.transFigure,
        clip_on=False
    )
    fig.add_artist(rect)

# ---------- 6. Save ----------
fig.suptitle("The Heat Is On: Wildfires and Conflict in Indonesia",
             color="black", fontsize=18, fontweight="bold", y=1.02)
plt.savefig("output/indonesia_maps.png", dpi=300, bbox_inches="tight",
            facecolor=fig.get_facecolor())
print("Saved: output/indonesia_maps.png")
plt.close()

Loading shapefiles...
Loading fire data...
  Fire records: 1,307,603
Loading ACLED data...
  WARNING unmatched: ['Bangka-Belitung']
  Provinces matched: 32 / 34
Plotting...


/var/folders/55/dqmn2ngx28xd2c95p7r6b9bm0000gn/T/ipykernel_4674/164999035.py:149: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  no_data_patch = mpatches.Patch(color="#eeeeee", label="No conflict recorded",


Saved: output/indonesia_maps.png
